In [8]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import os
import sys
from sklearn.model_selection import train_test_split
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu" )
print(device)

cuda:3


In [9]:
class DF_TJU():
    def __init__(self,args):
        self.normalization = True
        self.normalization_method = args.normalization_method # min-max, z-score
        self.args = args

    def _3_sigma(self, Ser1):
        rule = (Ser1.mean() - 3 * Ser1.std() > Ser1) | (Ser1.mean() + 3 * Ser1.std() < Ser1)
        index = np.arange(Ser1.shape[0])[rule]
        return index

    def delete_3_sigma(self,df):
        df = df.replace([np.inf, -np.inf], np.nan)
        df = df.dropna()
        df = df.reset_index(drop=True)
        out_index = []
        for col in df.columns:
            index = self._3_sigma(df[col])
            out_index.extend(index)
        out_index = list(set(out_index))
        df = df.drop(out_index, axis=0)
        df = df.reset_index(drop=True)
        return df

    def read_one_csv(self,file_name,nominal_capacity=None):
        df = pd.read_csv(file_name)
        df.insert(df.shape[1]-1,'cycle index',np.arange(df.shape[0]))

        df = self.delete_3_sigma(df)

        if nominal_capacity is not None:
            #print(f'nominal_capacity:{nominal_capacity}, capacity max:{df["capacity"].max()}',end=',')
            df['capacity'] = df['capacity']/nominal_capacity
            #print(f'SOH max:{df["capacity"].max()}')
            f_df = df.iloc[:,:-1]
            if self.normalization_method == 'min-max':
                f_df = 2*(f_df - f_df.min())/(f_df.max() - f_df.min()) - 1
            elif self.normalization_method == 'z-score':
                f_df = (f_df - f_df.mean())/f_df.std()

            df.iloc[:,:-1] = f_df

        return df

    def load_one_battery(self,path,nominal_capacity=None):
        df = self.read_one_csv(path,nominal_capacity)
        # TJU数据特征选择
        # Var2：'CV charge time','cycle index'
        # Var3：'CV charge time','current entropy','cycle index'
        df = df.filter(items=['CV charge time','current entropy','cycle index','capacity']) # CC Q在恒流充电时等价于CC charge time
        x = df.iloc[:,:-1].values
        y = df.iloc[:,-1].values
        x1 = x[:-1]
        x2 = x[1:]
        y1 = y[:-1]
        y2 = y[1:]
        return (x,y),(x1,y1),(x2,y2)

    def load_all_battery(self,path_list,nominal_capacity):
        X, Y, X1, X2, Y1, Y2 = [], [], [], [], [], []
        for path in path_list:
            (x, y),(x1, y1), (x2, y2) = self.load_one_battery(path, nominal_capacity)
            X.append(x)
            X1.append(x1)
            X2.append(x2)
            Y.append(y)
            Y1.append(y1)
            Y2.append(y2)

        X = np.concatenate(X, axis=0)
        X1 = np.concatenate(X1, axis=0)
        X2 = np.concatenate(X2, axis=0)
        Y = np.concatenate(Y, axis=0)
        Y1 = np.concatenate(Y1, axis=0)
        Y2 = np.concatenate(Y2, axis=0)

        tensor_X = torch.from_numpy(X).float().to(device) 
        tensor_X1 = torch.from_numpy(X1).float().to(device)
        tensor_X2 = torch.from_numpy(X2).float().to(device)
        tensor_Y = torch.from_numpy(Y).float().view(-1,1).to(device)
        tensor_Y1 = torch.from_numpy(Y1).float().view(-1,1).to(device)
        tensor_Y2 = torch.from_numpy(Y2).float().view(-1,1).to(device)

        train_X1, valid_X1, train_X2, valid_X2, train_Y1, valid_Y1, train_Y2, valid_Y2 = \
            train_test_split(tensor_X1, tensor_X2, tensor_Y1, tensor_Y2, test_size=0.2, random_state=420)
        train_loader = DataLoader(TensorDataset(train_X1, train_X2, train_Y1, train_Y2),
                                  batch_size=self.args.batch_size,
                                  shuffle=True)
        valid_loader = DataLoader(TensorDataset(valid_X1, valid_X2, valid_Y1, valid_Y2),
                                  batch_size=self.args.batch_size,
                                  shuffle=True)
        test_loader = DataLoader(TensorDataset(tensor_X1, tensor_X2, tensor_Y1, tensor_Y2),
                                 batch_size=self.args.batch_size,
                                 shuffle=False)

        data = {'input': tensor_X, 'label': tensor_Y,
                  'train_loader': train_loader,
                  'valid_loader': valid_loader,
                  'test_loader': test_loader}

        return data

In [10]:
class TJUdataFilter(DF_TJU):
    def __init__(self,root='../data/TJU data',args=None):
        super(TJUdataFilter, self).__init__(args)
        self.root = root
        self.batchs = ['Dataset_1_NCA_battery','Dataset_2_NCM_battery','Dataset_3_NCM_NCA_battery']
        if self.normalization:
            self.nominal_capacities = [3.5,3.5,2.5]
        else:
            self.nominal_capacities = [None,None,None]

    def read_one_batch(self,batch):
        assert batch in [1,2,3], 'batch must be in {}'.format([1,2,3])
        root = os.path.join(self.root,self.batchs[batch-1])
        file_list = os.listdir(root)
        df = pd.DataFrame()
        path_list = []
        for file in file_list:
            file_name = os.path.join(root,file)
            path_list.append(file_name)
        return self.load_all_battery(path_list=path_list, nominal_capacity=self.nominal_capacities[batch])  #?????batch-1

    def read_all(self,specific_path_list):
        for i,batch in enumerate(self.batchs):
            if batch in specific_path_list[0]:
                normal_capacity = self.nominal_capacities[i]
                break
        return self.load_all_battery(path_list=specific_path_list, nominal_capacity=normal_capacity)

In [11]:
def load_TJU_data_filter(args,small_sample=None):   # 无差分，差分：diff
    root = '../data/TJU data'
    data = TJUdataFilter(root=root, args=args)
    train_list = []
    test_list = []
    test_id = [['CY25-025_1-#5','CY25-05_1-#10','CY25-05_1-#16','CY25-05_1-#2','CY25-05_1-#8','CY25-1_1-#3','CY25-1_1-#9',
                'CY45-05_1-#1','CY45-05_1-#15','CY45-05_1-#19','CY45-05_1-#24','CY45-05_1-#28','CY45-05_1-#8'],# batch1
                ['CY25-05_1-#12','CY25-05_1-#16','CY25-05_1-#21','CY25-05_1-#4','CY35-05_1-#1','CY45-05_1-#1',
                 'CY45-05_1-#15','CY45-05_1-#19','CY45-05_1-#24','CY45-05_1-#28','CY45-05_1-#8'],# batch2
                 ['CY25-05_2-#2','CY25-05_4-#3']]# batch3
    batchs = os.listdir(root)
    batch = batchs[args.batch]
    batch_root = os.path.join(root,batch)
    files = os.listdir(batch_root)
    for f in files:
        if f[:-4] in test_id[args.batch]:
            test_list.append(os.path.join(batch_root,f))
            print(f)
        else:
            train_list.append(os.path.join(batch_root,f))
    if small_sample is not None:
        train_list = train_list[:small_sample]
    train_data = data.read_all(specific_path_list=train_list)
    test_data = data.read_all(specific_path_list=test_list)
    
    dataset = {'train_input':train_data['input'],
                  'train_label':train_data['label'],
                  'test_input':test_data['input'],
                  'test_label':test_data['label'],
                  'train_loader':train_data['train_loader'],
                  'valid_loader':train_data['valid_loader'],
                  'test_loader':test_data['test_loader']}
    return dataset

In [12]:
import argparse
parser = argparse.ArgumentParser('存储过滤出关键健康指标的HUST数据集')
parser.add_argument('--dataset',type=str,default='TJU',choices=['XJTU','HUST','MIT','TJU'])
parser.add_argument('--data_root', type=str, default='../data/TJU data', help='HUST数据集根路径')
parser.add_argument('--batch',type=int,default=2,choices=[0,1,2])
parser.add_argument('--normalization_method',type=str, default='min-max', help='min-max,z-score')
parser.add_argument('--batch_size',type=int,default=512)
args, _ = parser.parse_known_args()

In [13]:
dataset = load_TJU_data_filter(args)
torch.save(dataset, '../Data/TJU_Data_Var3_batch_2.pt')

CY25-05_4-#3.csv
CY25-05_2-#2.csv


/tmp/ipykernel_1940592/2408337709.py:41: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0     -1.000000
1     -0.997845
2     -0.995690
3     -0.993534
4     -0.991379
         ...   
924    0.991379
925    0.993534
926    0.995690
927    0.997845
928    1.000000
Name: cycle index, Length: 929, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.iloc[:,:-1] = f_df
/tmp/ipykernel_1940592/2408337709.py:41: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0     -1.00000
1     -0.99774
2     -0.99548
3     -0.99322
4     -0.99096
        ...   
881    0.99096
882    0.99322
883    0.99548
884    0.99774
885    1.00000
Name: cycle index, Length: 886, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.iloc[:,:-1] = f_df
/tmp/ipykernel_1940592/

In [14]:
import gc
torch.cuda.empty_cache()
gc.collect()

82